# Receipt image to structured JSON

A two-call example: read the image, then use LangChain to parse the response into JSON. Configure the root `.env` first. Running request cells sends data to Azure and may incur charges. Outputs have been cleared; no new cloud run has been recorded.

In [ ]:
from pathlib import Path
import sys

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "ai_lab.py").is_file()), None)
if ROOT is None:
    raise RuntimeError("Open this notebook from inside the repository folder.")
sys.path.insert(0, str(ROOT))
from ai_lab import chat, image_messages, transcribe_audio, synthesize_speech

In [ ]:
import importlib.util
spec = importlib.util.spec_from_file_location("receipt_cli", ROOT / "01_vision/receipt_analyzer/receipt_analysis.py")
receipt_cli = importlib.util.module_from_spec(spec)
spec.loader.exec_module(receipt_cli)
from ai_lab import AzureSettings
from langchain_openai import AzureChatOpenAI
settings = AzureSettings.load("text")
llm = AzureChatOpenAI(azure_endpoint=settings.endpoint, api_key=settings.key,
    api_version=settings.api_version, azure_deployment=settings.deployment, timeout=60, max_retries=2)

In [ ]:
image_path = ROOT / "01_vision/receipt_analyzer/test_receipt.jpg"
raw = chat(image_messages("Extract all visible receipt fields; leave missing fields unknown.", image_path))
print(raw)

In [ ]:
result = receipt_cli.get_response(raw, llm)
rows = receipt_cli.make_csv_rows(result)
result

The bundled `receipt_result.json` and `.csv` are supplied historical examples for `test_receipt.jpg`, not results generated by this notebook. Check extracted quantities, totals, and missing fields against the image before using them. The CLI writes new files under `generated/`.